# Apply an IRIS-AIA alignment result

The batch workflow in `iris_aia_batch_alignment.ipynb` stores one row per aligned frame pair in a TSV file. This notebook selects one result, applies its fitted rotation and translation to the corresponding IRIS frame, and compares the frame with its AIA reference.

The default selection reproduces the observation and IRIS frame used for the visual IRIS-AIA comparison in Figure 3.2 of the associated MSc thesis. It was chosen because the improvement is clearly visible in both the contour and color overlays. The FITS observations and alignment result table are not included in the repository; generate the TSV with the batch notebook and update the local data path below before running this example.

In [ ]:
import sys
from pathlib import Path

import irisreader as ir
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import rotate, shift
from sunpy.map import Map

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or examples directory.")
sys.path.insert(0, str(PROJECT_ROOT))

from utils import crop_frame, iris_to_sunpy_map

## Configuration

Select an observation and IRIS frame that occur exactly once in the alignment result table. The default is IRIS frame 1558 at approximately `2014-06-12T21:03:52`, as used in the thesis comparison. The matching AIA file is selected automatically from the frame index stored in the result row.

To generate this single-frame result with the batch notebook, use the following observation-list row and set `MAX_TIME_DIFFERENCE_SECONDS = 2`:

| obsid | start_time | end_time |
|---|---|---|
| 20140612_184427_3863605329 | 2014-06-12T21:03:52 | 2014-06-12T21:03:53 |

In [ ]:
OBSERVATION_ID = "20140612_184427_3863605329"
IRIS_FRAME = 1558

# Expected layout: DATA_DIR / OBSERVATION_ID / IRIS and AIA FITS files
DATA_DIR = PROJECT_ROOT / "data"
IRIS_OBSERVATION_DIR = DATA_DIR / OBSERVATION_ID
ALIGNMENT_RESULTS = PROJECT_ROOT / "outputs/iris_aia_alignment.tsv"

SAVE_FIGURE = False
FIGURE_PATH = PROJECT_ROOT / "outputs/iris_aia_alignment_example.png"

## Select the fitted parameters

The result row contains the matched frame numbers, timestamps, fitted shift, rotation, uncertainties, fit metric, and status. Pixel shifts are used below because the transformation is applied after IRIS has been reprojected onto the AIA pixel grid. The stored shifts describe the displacement measured by cross-correlation; applying the correction therefore uses the opposite sign.

In [ ]:
results = pd.read_csv(ALIGNMENT_RESULTS, sep="\t")
required_columns = {
    "Observation ID", "AIA frame", "IRIS frame", "AIA time", "IRIS time",
    "Shift pixel x", "Shift pixel y", "Shift arcsec x", "Shift arcsec y",
    "Angle", "RMSE or Chi2", "Status",
}
missing_columns = required_columns.difference(results.columns)
if missing_columns:
    raise ValueError(f"Alignment table is missing columns: {sorted(missing_columns)}")

selected = results.loc[
    (results["Observation ID"].astype(str) == OBSERVATION_ID)
    & (pd.to_numeric(results["IRIS frame"], errors="coerce") == IRIS_FRAME)
]

if len(selected) != 1:
    raise ValueError(
        f"Expected exactly one result for {OBSERVATION_ID}, IRIS frame {IRIS_FRAME}; "
        f"found {len(selected)}."
    )

result = selected.iloc[0]
if result["Status"] != "OK":
    raise ValueError(f"Selected alignment did not succeed: {result['Status']}")

aia_frame = int(result["AIA frame"])
dx_pixels = float(result["Shift pixel x"])
dy_pixels = float(result["Shift pixel y"])
angle_deg = float(result["Angle"])

summary_columns = [
    "AIA time", "IRIS time", "Shift arcsec x", "Shift arcsec y",
    "Angle", "RMSE or Chi2", "Status",
]
result[summary_columns]

## Apply the alignment

First, the IRIS map is rotated to solar north, the AIA frame is cropped to its field of view, and IRIS is reprojected onto the AIA grid. The fitted trial rotation is then applied. The stored cross-correlation shifts describe the measured displacement, so the image transformation uses the opposite sign in `(y, x)` array order.

In [ ]:
if not IRIS_OBSERVATION_DIR.is_dir():
    raise FileNotFoundError(f"IRIS observation directory not found: {IRIS_OBSERVATION_DIR}")
aia_files = sorted(IRIS_OBSERVATION_DIR.glob("*.image.fits"))
if not aia_files:
    raise FileNotFoundError(
        f"No AIA files matching *.image.fits found in {IRIS_OBSERVATION_DIR}"
    )
if not 0 <= aia_frame < len(aia_files):
    raise IndexError(
        f"AIA frame {aia_frame} is outside the available range 0-{len(aia_files) - 1}."
    )

observation = ir.observation(str(IRIS_OBSERVATION_DIR))
iris_sji = observation.sji[0]
iris_map = iris_to_sunpy_map(iris_sji, IRIS_FRAME).rotate(order=3, missing=-200)
aia_map = Map(aia_files[aia_frame])
cropped_aia = crop_frame(aia_map, iris_map)
iris_reprojected = iris_map.reproject_to(cropped_aia.wcs)

iris_rotated = rotate(
    iris_reprojected.data, angle=angle_deg, reshape=False, order=1
)
iris_aligned = shift(
    iris_rotated, shift=(-dy_pixels, -dx_pixels),
    order=1, mode="constant", cval=0,
)

In [ ]:
def display_normalize(data, low=1, high=99):
    """Normalize an image for display without modifying the source array."""
    data = np.asarray(data, dtype=float)
    finite = np.isfinite(data)
    if not np.any(finite):
        return np.zeros_like(data)
    vmin, vmax = np.percentile(data[finite], (low, high))
    if np.isclose(vmin, vmax):
        return np.zeros_like(data)
    return np.nan_to_num(np.clip((data - vmin) / (vmax - vmin), 0, 1))


def make_color_overlay(aia_data, iris_data):
    """Render AIA in orange and IRIS in blue; shared bright features become dark."""
    aia = display_normalize(aia_data)
    iris = display_normalize(iris_data)
    rgb = np.ones((*aia.shape, 3))
    rgb[..., 1] -= 0.5 * aia
    rgb[..., 2] -= aia
    rgb[..., 0] -= iris
    rgb[..., 1] -= 0.5 * iris
    return np.clip(rgb, 0, 1)

## Before and after

The top row overlays IRIS contours on AIA, making the displacement of individual bright structures easy to inspect. The bottom row combines both normalized images: AIA is orange, IRIS is blue, and spatially coincident bright structures appear dark. All fitted values shown in the title are read from the TSV rather than hard-coded.

In [ ]:
aia_display = display_normalize(cropped_aia.data)
contour_levels = np.nanpercentile(iris_reprojected.data, (85, 93, 98))

figure, axes = plt.subplots(
    2, 2, figsize=(11, 10), subplot_kw={"projection": cropped_aia.wcs}
)

for axis, iris_data, title in (
    (axes[0, 0], iris_reprojected.data, "Before alignment: contours"),
    (axes[0, 1], iris_aligned, "After alignment: contours"),
):
    axis.imshow(aia_display, cmap="gray_r", origin="lower")
    axis.contour(iris_data, levels=contour_levels, colors="red", linewidths=0.6)
    axis.set_title(title)

axes[1, 0].imshow(make_color_overlay(cropped_aia.data, iris_reprojected.data), origin="lower")
axes[1, 0].set_title("Before alignment: color overlay")
axes[1, 1].imshow(make_color_overlay(cropped_aia.data, iris_aligned), origin="lower")
axes[1, 1].set_title("After alignment: color overlay")

for axis in axes.flat:
    axis.set_xlabel("Solar X [arcsec]")
    axis.set_ylabel("Solar Y [arcsec]")

figure.suptitle(
    f"IRIS-AIA alignment: {OBSERVATION_ID}, IRIS frame {IRIS_FRAME}\n"
    f"Δx={float(result['Shift arcsec x']):+.2f} arcsec, "
    f"Δy={float(result['Shift arcsec y']):+.2f} arcsec, "
    f"θ={angle_deg:+.2f}°"
)
figure.tight_layout()

if SAVE_FIGURE:
    FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURE_PATH, dpi=200, bbox_inches="tight")
    print(f"Saved figure to {FIGURE_PATH.resolve()}")